# Module 02 — Organisational structural change

**Which firms, which plants, which locations — and who lost ground to whom?**

Module 01 asked *where* employment went, from exhaustive regional statistics. This module asks *inside which firms and plants* it moved, using the micro panel this thesis built — the level Eurostat cannot reach, and that Pavlínek (2019) could only approach through restructuring **events** and interviews. We do it on plant-level **stocks**.

Pavlínek's fourth research question is explicitly about the **ownership and nationality** of the firms driving job creation and loss. Section 4 is the direct analogue, measured rather than inferred.

## ⚠️ Read the universe before reading any result

Headline figures use the **balanced imputed panel**, assembly + component, strictly balanced over **2012–2023 → 31 plants**. That universe contains **only the three German groups**:

| | plants |
|---|---|
| Volkswagen Group | 17 |
| Mercedes-Benz | 12 |
| BMW Group | 5 |
| *of which in Germany / abroad* | *27 / 7* |

So this module does **not** describe European automotive firms in general. It describes, without coverage bias, **German automotive capital** — how it restructured at home while expanding into the integrated periphery. Narrower than Module 01, but sharper: Pavlínek's mechanism seen from inside the firms driving it.

Renault, Stellantis, Ford and Toyota appear **only** in the full-universe context figures (Section 8), which must not be read as trends.

In [ ]:
# Reload edited .py modules automatically — without this, a kernel that
# imported an older version keeps serving it (AttributeError on new functions).
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import foundation as fd
import module02_organisational as m2
# --- archive bootstrap: resolve paths from the project root ---------------
# The original was CONFIG.data_dir = './data', which assumed the notebook ran
# beside a data/ folder. In the archived layout the inputs live in
# 02_data_inputs/, so the path is taken from specular_paths instead and this
# notebook works from wherever it is opened.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))   # 01_code/
from specular_paths import P
fd.CONFIG.data_dir = str(P.inputs)
fd.CONFIG.out_dir  = str(P.outputs)
print('data:', fd.CONFIG.data_dir)
fd.CONFIG.out_dir  = './outputs'
pd.set_option('display.width', 200)

## 1. The universe — and why `strict` balancing is not optional

“Balanced” in Module 00 means a complete 35 × 20 **grid**. It does **not** mean a constant set of contributing plants: `inactive` cells carry NaN and drop out, so over 2006–2025 the contributing count runs **23 (2006) → 34 (2019–23) → 25 (2025)**.

Summing across that produces a spurious rise and a cliff at both ends. In an early draft of this module it manufactured two false findings — Mercedes employment appearing to **halve**, and component employment appearing to **collapse ~70% in 2025**. Both were plants leaving the sample. Neither survives strict balancing.

In [ ]:
u = m2.analysis_universe()          # strict, 2012–2023, assembly+component
print('plants:', u.attrs['n_plants'], '| window:', u.attrs['window'], '| strict:', u.attrs['strict'])
print('contributing plants per year:', u.groupby('year')['plant_uid'].nunique().unique(), '<- constant')
print('groups:', u.groupby('group')['plant_uid'].nunique().to_dict())

# what the unbalanced version would have looked like
raw = m2.analysis_universe(window=None, strict=False)
print('\nunbalanced contributing plants per year:')
print(raw.groupby('year')['plant_uid'].nunique().to_string())

## 2. Group recomposition

Levels carry ±1.96·SE bands propagated from the per-cell imputation SD (delta method). The bands are a **lower bound** — they assume cells are independent, which they are not.

In [ ]:
gt = m2.group_trajectories(u)
fig = m2.fig_group_trajectories(gt, save=str(fd.CONFIG.out('fig02_group.png'))); plt.show()
display(gt.pivot(index='year', columns='group', values='share_pct').round(1))

## 3. Assembly vs component

Where the powertrain transition should bite hardest — engine, transmission and component sites versus vehicle assembly.

In [ ]:
ct = m2.classification_trajectories(u)
fig = m2.fig_assembly_component(ct, save=str(fd.CONFIG.out('fig02_assembly_component.png'))); plt.show()
display(ct.pivot(index='year', columns='classification', values='index_100').round(1))

## 4. Pavlínek's axis — home vs periphery

All three groups are German-owned, so `location_type` reduces cleanly to **Germany (home)** vs **periphery (abroad)** — 27 vs 7 plants (BE, ES, HU×2, PL, PT, SK). German capital observed from the inside.

Ownership coding handles Stellantis by **brand heritage** (PSA→FR, Fiat→IT, Opel→DE) rather than the post-2021 merged entity — relevant only to the context figures, since no Stellantis plant is in this universe.

In [ ]:
hp = m2.home_vs_periphery(u)
fig = m2.fig_home_periphery(hp, save=str(fd.CONFIG.out('fig02_home_periphery.png'))); plt.show()

own = m2.ownership_coding()
print('ownership coding source:', own['home_country_source'].value_counts().to_dict())
print('home country (full universe):', own['home_country'].value_counts().to_dict())

## 5. Within-group reallocation — the intra-firm spatial fix

$$\Delta E_p = \underbrace{E_{p0}\,g_G}_{\text{group trend}} + \underbrace{E_{p0}(g_p-g_G)}_{\text{reallocation}}$$

Positive reallocation = the plant gained work **relative to its own parent group** — a stronger claim than gaining relative to the industry. Note Ingolstadt: it *grew* in absolute terms yet reallocates negatively, because VW grew faster.

In [ ]:
wg = m2.within_group_reallocation(u, 2012, 2023)
fig = m2.fig_within_group(wg, save=str(fd.CONFIG.out('fig02_within_group.png'))); plt.show()
cols = ['plant_name','country_iso2','group','E0','E1','growth_rate','reallocation']
print('GAINED relative to own group'); display(wg.head(8)[cols].round(2))
print('LOST relative to own group');   display(wg.tail(8)[cols].round(2))
wg.to_csv('outputs/tbl02_within_group_reallocation.csv', index=False)

## 6. Plant concentration

Is work consolidating into fewer, larger plants, or spreading across the network? Complements Module 01's *regional* concentration.

In [ ]:
pc = m2.plant_concentration(u)
fig = m2.fig_plant_concentration(pc, save=str(fd.CONFIG.out('fig02_plant_concentration.png'))); plt.show()
display(pc.round(1))

## 7. Data availability — where the panel is evidence, and where it is silence

**This is not a plant operating record.** An earlier version of this function read the panel's `inactive` flags as plant entry/exit. That was wrong: for every plant, `inactive` years sit exactly **outside its observed span** (VW Kassel inactive 2006–2018 / observed 2019–2025; Untertürkheim inactive 2025 / observed 2014–2024). Kassel has run since 1958; Untertürkheim still operates.

`inactive` marks where the imputation declined to extrapolate beyond the evidence — a boundary of **knowledge**, not of existence. Genuine closures must come from the registry (`classification == 'historic'`) or external verification.

In [ ]:
tl = m2.data_availability_window()
fig = m2.fig_data_availability(tl, save=str(fd.CONFIG.out('fig02_data_availability.png'))); plt.show()
print('plants with full 2006–2025 support: %d / %d' % ((tl['coverage_pct']==100).sum(), len(tl)))
display(tl.sort_values('coverage_pct')[['plant_name','country_iso2','first_supported_year','last_supported_year','coverage_pct']].head(10))

## 8. Full universe — context only

All 89 plants, observed data. Renault / Stellantis / Ford / Toyota appear here and nowhere else. **Panel (a) must be read with panel (b):** rising lines largely reflect improving coverage, not growth.

In [ ]:
fig = m2.fig_full_universe_context(save=str(fd.CONFIG.out('fig02_full_universe_context.png'))); plt.show()

---
## What survives strict balancing (31 plants, 2012–2023)

| finding | 2012 | 2023 |
|---|---|---|
| VW Group share | 61.3% | **65.8%** |
| Mercedes-Benz share | 24.7% | 21.6% |
| BMW share | 14.0% | 12.6% |
| employment abroad | 35k | **50k** |
| employment at home (DE) | 265k | 294k |
| **share located abroad** | 11.6% | **14.6%** |
| assembly (index) | 100 | 114.8 |
| component (index) | 100 | 113.8 |

**The robust story.** Volkswagen consolidated its share of German automotive employment; employment abroad grew ~43% against ~11% at home, lifting the foreign share from 11.6% to 14.6%. That is Pavlínek's peripheral expansion, visible inside the firms — but note it is *expansion abroad alongside modest growth at home*, not hollowing out. Within groups, work moved toward Poznań, Kecskemét, Győr and Landshut, and away from Ingolstadt, Neckarsulm, Munich and Sindelfingen relative to their own parents.

**Two findings that did NOT survive** (both were sample-composition artefacts, now corrected): Mercedes employment *halving*, and component employment *collapsing*. Over the strictly balanced window Mercedes is broadly flat and assembly and component grow at similar rates.

## To decide before writing

1. **Window.** 2012–2023 gives 31 plants; 2014–2023 would give 33; 2006–2025 only 20. Trade length against width.
2. **Mercedes volatility.** The indexed path is saw-toothed — likely imputation volatility on sparsely observed plants. Worth checking against the per-plant bands before commenting on it.
3. **Scope.** Currently assembly + component. Splitting them was informative; consider whether `component` deserves its own sub-analysis given the electrification angle.
4. **Uncertainty.** If the bands ever carry an argument, recompute them from the M completed panels with Rubin's rules rather than the delta-method lower bound.

**Next — Module 03: product / size structural change** (kerb weight, segment, EV share) linking the model mix to employment.